# Verify worker-timeout recovery on Fabric

`recover_worker_timeouts` restarts the CPython worker after a timeout instead of
ending the run. It passes 25 tests locally, but three things can only be checked
on a real Fabric capacity:

1. whether a worker subprocess can be **respawned** inside the Synapse executor,
2. whether **Lakehouse `File` inputs** re-bind after the restart,
3. whether **`FabricLM`** keeps working across a restart (it refreshes an AAD
   token, and the restart happens mid-run).

Each cell prints `PASS` or `FAIL`. Run top to bottom and send back the output.

Install the branch under test:
```
%pip install git+https://github.com/pawarbi/fabric-rlm-core.git@feat/harness-resilience
```

In [ ]:
%pip install -q git+https://github.com/pawarbi/fabric-rlm-core.git@feat/harness-resilience

In [ ]:
import inspect, pathlib
from fabric_rlm import RLM, File

RESULTS = {}

def check(name, ok, detail=""):
    RESULTS[name] = bool(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f"  --  {detail}" if detail else ""))

has_kwarg = "recover_worker_timeouts" in inspect.signature(RLM.__init__).parameters
check("branch installed", has_kwarg,
      "if FAIL, the %pip install did not take -- restart the session and rerun")

LAKEHOUSE = pathlib.Path("/lakehouse/default/Files")
print("\nLakehouse mounted:", LAKEHOUSE.exists())

In [ ]:
# A scripted LM keeps this test about the runtime, not model behaviour.
class ScriptedLM:
    def __init__(self, turns):
        self.turns, self.i = list(turns), 0
    def __call__(self, messages=None, prompt=None, **kwargs):
        turn = self.turns[min(self.i, len(self.turns) - 1)]
        self.i += 1
        return [turn]

def code(body):
    return f"```python\n{body}\n```"

HANG = code("import time\ntime.sleep(60)")
print("helpers ready")

## 1. Does the worker respawn inside the Synapse executor?

In [ ]:
res = RLM.task(
    task="t", outputs=["answer"],
    lm=ScriptedLM([HANG, code("SUBMIT(answer='recovered')")]),
    max_turns=4, timeout=5, recover_worker_timeouts=1,
).run()

check("worker respawns after timeout",
      res.submitted and res.payload.get("answer") == "recovered",
      f"submitted={res.submitted} failure_reason={getattr(res, 'failure_reason', None)}")

turns = [t for t in res.trajectory if getattr(t, "code", None)]
check("earlier turns are kept", len(turns) >= 2, f"{len(turns)} turns recorded")

## 2. Do Lakehouse `File` inputs re-bind after the restart?

This is the one that matters in practice: the worker is replaced, so the input
binding has to be reapplied against the mounted path.

In [ ]:
base = LAKEHOUSE if LAKEHOUSE.exists() else pathlib.Path.cwd()
probe = base / "rlm_recovery_probe.csv"
probe.write_text("a,b\n1,2\n3,4\n5,6\n", encoding="utf-8")
print("probe written to", probe)

res = RLM.task(
    task="t", inputs={"table": File(probe)}, outputs=["answer"],
    lm=ScriptedLM([
        HANG,
        code("import csv\n"
             "rows = list(csv.reader(open(table.path)))\n"
             "SUBMIT(answer=str(len(rows)))"),
    ]),
    max_turns=4, timeout=5, recover_worker_timeouts=1,
).run()

check("Lakehouse File input re-binds after restart",
      res.submitted and res.payload.get("answer") == "4",
      f"answer={res.payload.get('answer') if res.payload else None} (expected '4')")

## 3. Does `FabricLM` survive a restart mid-run?

The LM lives in the notebook process, not the worker, so this should be
unaffected -- but `FabricLM` refreshes an AAD token on long runs and the restart
lands in the middle, so it is worth confirming with a real model call.

In [ ]:
try:
    from fabric_rlm import FabricLM
    lm = FabricLM("gpt-5.1")

    res = RLM.task(
        task=("First run exactly this and nothing else: `import time; time.sleep(60)`.\n"
              "It will time out and the worker will restart. After that, compute "
              "2+2 in Python and SUBMIT(answer=<the number as a string>)."),
        outputs=["answer"], lm=lm,
        max_turns=6, timeout=5, recover_worker_timeouts=1,
    ).run()

    check("FabricLM works across a restart",
          res.submitted and "4" in str(res.payload.get("answer", "")),
          f"answer={res.payload.get('answer') if res.payload else None}")

    timed_out = [t for t in res.trajectory
                 if "timed out" in str(getattr(t, "error", "") or "").lower()]
    check("a real timeout actually occurred", bool(timed_out),
          "if FAIL the model may not have run the sleep, so recovery was untested")
except Exception as exc:
    check("FabricLM works across a restart", False, f"{type(exc).__name__}: {exc}")

## 4. No leaked workers

A notebook kernel is long-lived. If each recovery orphaned a subprocess, a
session doing repeated analyses would accumulate them.

In [ ]:
try:
    import os, psutil
    me = psutil.Process(os.getpid())
    before = len(me.children(recursive=True))
    for _ in range(3):
        RLM.task(task="t", outputs=["answer"],
                 lm=ScriptedLM([HANG, code("SUBMIT(answer='ok')")]),
                 max_turns=4, timeout=5, recover_worker_timeouts=1).run()
    after = len(me.children(recursive=True))
    check("no worker leak across 3 recoveries", after <= before + 1,
          f"{before} children before, {after} after")
except ImportError:
    print("SKIP  psutil not available")

## 5. Opting out still fails fast

The default is one recovery. Passing `recover_worker_timeouts=0` restores the
old behaviour, where a timeout ends the run immediately.

In [ ]:
res = RLM.task(task="t", outputs=["answer"],
               lm=ScriptedLM([HANG, code("SUBMIT(answer='should not reach')")]),
               max_turns=4, timeout=5, recover_worker_timeouts=0).run()

check("opting out still fails fast",
      (not res.submitted) and getattr(res, "failure_reason", None) == "worker_timeout",
      f"submitted={res.submitted} failure_reason={getattr(res, 'failure_reason', None)}")

# And the default (no kwarg) should now recover on its own.
res2 = RLM.task(task="t", outputs=["answer"],
                lm=ScriptedLM([HANG, code("SUBMIT(answer='default recovered')")]),
                max_turns=4, timeout=5).run()

check("default recovers without being asked",
      res2.submitted and res2.payload.get("answer") == "default recovered",
      f"submitted={res2.submitted}")

try:
    probe.unlink()
except Exception:
    pass

## Summary

In [ ]:
print(f"{sum(RESULTS.values())} of {len(RESULTS)} checks passed\n")
for name, ok in RESULTS.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

failed = [n for n, ok in RESULTS.items() if not ok]
print("\nAll good." if not failed else f"\nFailed: {failed}")